# Wine Weather Data Acquisition Pipeline

**Author:** [Your Name]  
**Last Updated:** 2026-05-09

## Overview
This notebook enriches Vivino wine rating data with historical growing-season weather data from the Open-Meteo Archive API.

## Data Sources
| Source | Description | License |
|--------|-------------|---------|
| [Vivino Wine Dataset](https://www.kaggle.com/datasets/joshuakalobbowles/vivino-wine-data-top-10-countries-exchina) | Wine ratings, prices, regions | Kaggle / original terms |
| [Open-Meteo Historical Archive](https://open-meteo.com/) | Daily temp, precipitation, humidity | **CC BY 4.0** |

## Features Added
- `growing_season_avg_temp_c` – Mean temperature during growing season (Apr–Oct Northern, Oct–Apr Southern)
- `growing_season_total_precip_mm` – Total precipitation during growing season
- `growing_season_avg_humidity_percent` – Mean relative humidity during growing season

## Rate Limit Compliance
- Open-Meteo free tier: **10,000 requests/day**
- This pipeline includes: exponential backoff on 429 errors, 1.5s delays between requests, checkpoint caching for resumability

## Attribution
Weather data © Open-Meteo (CC BY 4.0). Wine data from Kaggle user [joshuakalobbowles](https://www.kaggle.com/joshuakalobbowles).

## 1. Configuration

In [ ]:
# ============================================================
# CONFIGURATION - All settings in one place
# ============================================================

# Data file paths
INPUT_FILE = 'vivino_top_ten_with_region_weather.csv'
OUTPUT_FILE = 'vivino_top_ten_with_region_weather_filled.csv'
CHECKPOINT_FILE = 'weather_cache_checkpoint.pkl'

# Weather API settings
WEATHER_API_URL = "https://archive-api.open-meteo.com/v1/archive"
GEOCODING_API_URL = "https://geocoding-api.open-meteo.com/v1/search"
MAX_YEARS_PER_REQUEST = 15  # Open-Meteo can handle ~15 years efficiently
DELAY_BETWEEN_REQUESTS = 1.5  # Seconds between API calls (rate limiting)
MAX_RETRIES = 5
MIN_YEAR = 1940  # Open-Meteo historical data starts here

# Weather columns to fetch
WEATHER_COLS = [
    'growing_season_avg_temp_c',
    'growing_season_total_precip_mm',
    'growing_season_avg_humidity_percent'
]

# Hemisphere growing season date ranges
# Northern Hemisphere: April 1 - October 31
# Southern Hemisphere: October 1 (previous year) - April 30
SEASON_DATES = {
    'Northern': ('04-01', '10-31'),
    'Southern': ('10-01', '04-30')
}

# Country to ISO code mapping for geocoding
COUNTRY_TO_CODE = {
    'Argentina': 'AR', 'Australia': 'AU', 'Chile': 'CL',
    'France': 'FR', 'Germany': 'DE', 'Italy': 'IT',
    'Portugal': 'PT', 'South Africa': 'ZA', 'Spain': 'ES',
    'United States': 'US'
}

# Hemisphere mapping for each country
HEMISPHERE_MAP = {
    # Southern Hemisphere
    'Argentina': 'Southern', 'Australia': 'Southern', 'Chile': 'Southern',
    'South Africa': 'Southern',
    # Northern Hemisphere
    'Germany': 'Northern', 'Spain': 'Northern', 'France': 'Northern',
    'Italy': 'Northern', 'Portugal': 'Northern', 'United States': 'Northern'
}

## 2. Imports

In [ ]:
import math
import os
import pickle
import random
import time
from datetime import datetime

import pandas as pd
import requests

# Optional: Kaggle dataset download
try:
    import kagglehub
    KAGGLE_AVAILABLE = True
except ImportError:
    KAGGLE_AVAILABLE = False
    print("kagglehub not installed. Will use local CSV if available.")

## 3. Utility Functions

In [ ]:
def fetch_with_retry(url, params, max_retries=MAX_RETRIES):
    """
    Fetch data from API with exponential backoff for rate limits.
    
    Args:
        url: API endpoint
        params: Dictionary of query parameters
        max_retries: Maximum number of retry attempts
    
    Returns:
        Response object or None if failed
    """
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params=params, timeout=30)
            
            if resp.status_code == 429:  # Too Many Requests
                wait_time = (2 ** attempt) + random.uniform(0, 2)
                print(f"Rate limited. Waiting {wait_time:.1f}s... (attempt {attempt+1}/{max_retries})")
                time.sleep(wait_time)
                continue
            
            resp.raise_for_status()
            return resp
        
        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                print(f"Failed after {max_retries} attempts: {str(e)[:100]}")
                return None
            wait_time = (2 ** attempt) + random.uniform(0, 1)
            print(f"Request error: {str(e)[:50]}. Waiting {wait_time:.1f}s...")
            time.sleep(wait_time)
    
    return None


def geocode_region(region_name, country_name):
    """
    Geocode a wine region using Open-Meteo Geocoding API.
    
    Args:
        region_name: Name of the wine region
        country_name: Country name (used for filtering)
    
    Returns:
        Tuple of (latitude, longitude) or (None, None)
    """
    if not region_name or pd.isna(region_name):
        return None, None
    
    params = {
        "name": str(region_name).strip(),
        "count": 5,
        "language": "en",
        "format": "json"
    }
    
    country_code = COUNTRY_TO_CODE.get(country_name)
    if country_code:
        params["countryCode"] = country_code
    
    try:
        resp = fetch_with_retry(GEOCODING_API_URL, params)
        if resp and resp.status_code == 200:
            data = resp.json()
            if data.get("results"):
                r = data["results"][0]
                return float(r["latitude"]), float(r["longitude"])
    except Exception as e:
        print(f"Geocoding error for '{region_name}': {str(e)[:80]}")
    
    return None, None


def get_growing_season_dates(year, hemisphere):
    """
    Get start and end dates for growing season.
    
    Args:
        year: Vintage year
        hemisphere: 'Northern' or 'Southern'
    
    Returns:
        Tuple of (start_date, end_date) as pandas Timestamps
    """
    start_mo, end_mo = SEASON_DATES[hemisphere]
    
    if hemisphere == 'Northern':
        start_date = pd.Timestamp(f"{year}-{start_mo}")
        end_date = pd.Timestamp(f"{year}-{end_mo}")
    else:  # Southern: growing season spans Oct-Apr of year-1 to year
        start_date = pd.Timestamp(f"{year-1}-{start_mo}")
        end_date = pd.Timestamp(f"{year}-{end_mo}")
    
    return start_date, end_date


def fetch_weather_for_location(lat, lon, years, hemisphere):
    """
    Fetch weather data for a location across multiple years.
    
    Args:
        lat: Latitude
        lon: Longitude
        years: List of vintage years to fetch
        hemisphere: 'Northern' or 'Southern'
    
    Returns:
        Dictionary mapping (lat, lon, year) -> weather stats
    """
    # Filter to available years (Open-Meteo starts at 1940)
    years = [y for y in years if y >= MIN_YEAR]
    if not years:
        return {}
    
    years = sorted(years)
    weather_cache = {}
    
    for i in range(0, len(years), MAX_YEARS_PER_REQUEST):
        chunk_years = years[i:i + MAX_YEARS_PER_REQUEST]
        
        # Get overall date range for this chunk
        if hemisphere == 'Northern':
            start_date = f"{min(chunk_years)}-04-01"
            end_date = f"{max(chunk_years)}-10-31"
        else:
            start_date = f"{min(chunk_years)-1}-10-01"
            end_date = f"{max(chunk_years)}-04-30"
        
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": start_date,
            "end_date": end_date,
            "daily": "temperature_2m_mean,precipitation_sum,relative_humidity_2m_mean",
            "timezone": "auto"
        }
        
        resp = fetch_with_retry(WEATHER_API_URL, params)
        
        if resp is None:
            # Mark all years in chunk as missing
            for year in chunk_years:
                key = (round(lat, 4), round(lon, 4), year)
                weather_cache[key] = {col: None for col in WEATHER_COLS}
            continue
        
        data = resp.json()
        daily_df = pd.DataFrame({
            'date': pd.to_datetime(data['daily']['time']),
            'temp_mean': data['daily']['temperature_2m_mean'],
            'precip_sum': data['daily']['precipitation_sum'],
            'humidity_mean': data['daily']['relative_humidity_2m_mean']
        })
        
        for year in chunk_years:
            start_date, end_date = get_growing_season_dates(year, hemisphere)
            season = daily_df[(daily_df['date'] >= start_date) & (daily_df['date'] <= end_date)]
            
            if len(season) < 10:  # Insufficient data
                stats = {col: None for col in WEATHER_COLS}
            else:
                stats = {
                    'growing_season_avg_temp_c': round(season['temp_mean'].mean(), 2),
                    'growing_season_total_precip_mm': round(season['precip_sum'].sum(), 2),
                    'growing_season_avg_humidity_percent': round(season['humidity_mean'].mean(), 2)
                }
            
            key = (round(lat, 4), round(lon, 4), year)
            weather_cache[key] = stats
        
        time.sleep(DELAY_BETWEEN_REQUESTS)
    
    return weather_cache

## 4. Load Wine Dataset

In [ ]:
# Try to download from Kaggle, or load local file
if KAGGLE_AVAILABLE and not os.path.exists(INPUT_FILE):
    print("Downloading wine dataset from Kaggle...")
    wine_path = kagglehub.dataset_download('joshuakalobbowles/vivino-wine-data-top-10-countries-exchina')
    wine_df = pd.read_csv(os.path.join(wine_path, 'vivino_top_ten.csv'))
    print(f"Downloaded {len(wine_df)} rows")
elif os.path.exists(INPUT_FILE):
    wine_df = pd.read_csv(INPUT_FILE)
    print(f"Loaded {len(wine_df)} rows from {INPUT_FILE}")
else:
    raise FileNotFoundError("No wine dataset found. Please install kagglehub or provide the CSV file.")

# Add Hemisphere column
wine_df['Hemisphere'] = wine_df['Country'].map(HEMISPHERE_MAP)

# Clean Year column (remove N.V. and missing)
wine_df = wine_df[wine_df['Year'].notna() & (wine_df['Year'] != 'N.V.')].copy()
wine_df['Year'] = wine_df['Year'].astype(int)

print(f"After cleaning: {len(wine_df)} rows, {wine_df['Winery'].nunique()} unique wineries")

## 5. Geocode Regions (Get Coordinates)

In [ ]:
def geocode_all_regions(df):
    """Geocode unique region-country pairs and merge back."""
    unique_regions = df[['Region', 'Country']].drop_duplicates().dropna(subset=['Region'])
    print(f"Geocoding {len(unique_regions)} unique regions...")
    
    coords = []
    for _, row in unique_regions.iterrows():
        lat, lon = geocode_region(row['Region'], row['Country'])
        coords.append({'Region': row['Region'], 'Country': row['Country'], 
                       'Latitude': lat, 'Longitude': lon})
        time.sleep(0.35)  # Polite delay between geocoding requests
    
    coord_df = pd.DataFrame(coords)
    return df.merge(coord_df, on=['Region', 'Country'], how='left')

# Run geocoding (skip if coordinates already exist)
if 'Latitude' not in wine_df.columns:
    wine_df = geocode_all_regions(wine_df)
    
# Drop rows without coordinates
initial_rows = len(wine_df)
wine_df = wine_df.dropna(subset=['Latitude', 'Longitude']).copy()
print(f"Kept {len(wine_df)} rows with valid coordinates (dropped {initial_rows - len(wine_df)})")

## 6. Fetch Weather Data (with Checkpointing)

In [ ]:
# Load existing checkpoint if available
weather_cache = {}
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'rb') as f:
        weather_cache = pickle.load(f)
    print(f"Loaded {len(weather_cache)} cached weather records")

# Group by unique location
location_data = {}
for _, row in wine_df.iterrows():
    loc_key = (round(row['Latitude'], 4), round(row['Longitude'], 4))
    if loc_key not in location_data:
        location_data[loc_key] = {
            'years': set(),
            'hemisphere': row['Hemisphere'],
            'lat': row['Latitude'],
            'lon': row['Longitude']
        }
    location_data[loc_key]['years'].add(int(row['Year']))

print(f"Unique locations to query: {len(location_data)}")

# Determine which locations need fetching
locations_to_fetch = []
for loc_key, info in location_data.items():
    lat, lon = info['lat'], info['lon']
    years_needed = []
    for year in info['years']:
        cache_key = (round(lat, 4), round(lon, 4), year)
        if cache_key not in weather_cache:
            years_needed.append(year)
        elif weather_cache[cache_key]['growing_season_avg_temp_c'] is None:
            years_needed.append(year)  # Previously failed, retry
    
    if years_needed:
        locations_to_fetch.append({
            'lat': lat,
            'lon': lon,
            'years': years_needed,
            'hemisphere': info['hemisphere']
        })

print(f"Locations needing fetch: {len(locations_to_fetch)}")

# Fetch missing weather data
for i, loc in enumerate(locations_to_fetch):
    print(f"\n[{i+1}/{len(locations_to_fetch)}] Fetching ({loc['lat']:.4f}, {loc['lon']:.4f}) - {len(loc['years'])} years")
    new_data = fetch_weather_for_location(loc['lat'], loc['lon'], loc['years'], loc['hemisphere'])
    weather_cache.update(new_data)
    
    # Save checkpoint every 10 locations
    if (i + 1) % 10 == 0:
        with open(CHECKPOINT_FILE, 'wb') as f:
            pickle.dump(weather_cache, f)
        print(f"Checkpoint saved ({i+1}/{len(locations_to_fetch)} locations)")

# Final checkpoint
with open(CHECKPOINT_FILE, 'wb') as f:
    pickle.dump(weather_cache, f)
print(f"\nWeather cache saved with {len(weather_cache)} records")

## 7. Assign Weather Data to Wines

In [ ]:
# Initialize weather columns
for col in WEATHER_COLS:
    if col not in wine_df.columns:
        wine_df[col] = None

# Assign weather data
assigned = 0
still_missing = 0

for idx, row in wine_df.iterrows():
    key = (round(row['Latitude'], 4), round(row['Longitude'], 4), int(row['Year']))
    if key in weather_cache:
        stats = weather_cache[key]
        for col in WEATHER_COLS:
            wine_df.at[idx, col] = stats[col]
        if stats['growing_season_avg_temp_c'] is not None:
            assigned += 1
        else:
            still_missing += 1
    else:
        still_missing += 1

print(f"Assigned weather data: {assigned} wines")
print(f"Still missing weather data: {still_missing} wines")

## 8. Save Final Dataset

In [ ]:
wine_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nFinal dataset saved to '{OUTPUT_FILE}'")
print(f"Shape: {wine_df.shape}")
print(f"Columns: {list(wine_df.columns)}")

## 9. Quick Summary Statistics

In [ ]:
print("=" * 50)
print("Weather Data Coverage")
print("=" * 50)
for col in WEATHER_COLS:
    non_null = wine_df[col].notna().sum()
    print(f"{col}: {non_null}/{len(wine_df)} ({100*non_null/len(wine_df):.1f}%)")

print("\n" + "=" * 50)
print("Sample of Enriched Data")
print("=" * 50)
display_cols = ['Winery', 'Year', 'Region', 'Rating', 'Price'] + WEATHER_COLS
wine_df[display_cols].head(10)